<a href="https://colab.research.google.com/github/mochwendy/sentimen-app/blob/main/Portofolio_AI_Engineer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# 1. Install library pemrosesan audio dan visual
!pip install gradio librosa soundfile numpy -q
# Pastikan ffmpeg bawaan Google Colab siap digunakan
!apt-get install ffmpeg -y -q

import os
import subprocess
import numpy as np
import librosa
import gradio as gr

print("✅ Semua infrastruktur Audio AI & FFmpeg siap digunakan!")

# 2. FUNGSI AI LAYER: Ekstraksi dan Deteksi Audio
def analisis_audio_video(video_path):
    audio_temp = "audio_ekstrak.wav"

    # Bersihkan file sampah terdahulu jika ada
    if os.path.exists(audio_temp):
        os.remove(audio_temp)

    try:
        # Menggunakan FFmpeg untuk mengambil audio dari video dan mengubahnya ke format WAV
        perintah_ffmpeg = [
            'ffmpeg', '-i', video_path,
            '-vn', '-acodec', 'pcm_s16le',
            '-ar', '16000', '-ac', '1',
            audio_temp, '-y', '-loglevel', 'quiet'
        ]
        subprocess.run(perintah_ffmpeg, check=True)

        # Jika FFmpeg sukses tapi file tidak terbuat, berarti video tersebut MUTE (tidak ada track audio)
        if not os.path.exists(audio_temp) or os.path.getsize(audio_temp) == 0:
            return "MUTE (Hening Total)", 0.0, "OTOMATIS LOLOS (Tanpa Suara)"

        # Membaca sinyal audio digital menggunakan Librosa
        sinyal, sr = librosa.load(audio_temp, sr=16000)

        if len(sinyal) == 0:
            return "MUTE (Hening Total)", 0.0, "OTOMATIS LOLOS (Sinyal Kosong)"

        # A. Hitung desibel rata-rata untuk mendeteksi keheningan (Silence Detection)
        rms = librosa.feature.rms(y=sinyal)
        rata_rata_energi = np.mean(rms)

        # Ambang batas hening (jika energi sangat dekat dengan 0)
        if rata_rata_energi < 0.001:
            return "SILENT (Hening/Mute)", float(rata_rata_energi), "OTOMATIS LOLOS (Suara Tidak Terdengar)"

        # B. Audio Fingerprint Sederhana: Deteksi Ritme/Musik berbasis Spectral Flatness & Chroma
        # Musik biasanya memiliki pola nada (chroma) dan variasi frekuensi yang kaya dibanding noise/suara datar
        flatness = librosa.feature.spectral_flatness(y=sinyal)
        rata_rata_flatness = np.mean(flatness)

        # Jika nilai flatness rendah, berarti audio memiliki struktur frekuensi padat (indikasi musik/suara vokal jelas)
        status_ai = "TERDETEKSI AUDIO/MUSIK" if rata_rata_flatness < 0.1 else "TERDETEKSI BACKGROUND NOISE"
        keputusan_sistem = "PERLU TINJAUAN MANUAL ⚠️" if rata_rata_flatness < 0.1 else "OTOMATIS LOLOS ✅"

        return status_ai, float(rata_rata_flatness), keputusan_sistem

    except subprocess.CalledProcessError:
        # Jika FFmpeg gagal membaca track audio, indikasi kuat video tidak memiliki suara
        return "MUTE (Tidak ada jalur audio)", 0.0, "OTOMATIS LOLOS (No Audio Track)"

# 3. FUNGSI LAYER REVIEW MANUAL
# Database simulasi untuk menyimpan keputusan manusia
riwayat_review = []

def proses_review_manusia(video, keputusan_admin, catatan):
    if video is None:
        return "Silakan unggah video terlebih dahulu untuk dianalisis.", ""

    # Jalankan lapisan AI otomatis terlebih dahulu
    status_ai, skor, keputusan_sistem = analisis_audio_video(video)

    # Gabungkan keputusan AI dengan aksi ulasan manusia
    hasil_log = {
        "Nama File": os.path.basename(video),
        "Analisis AI": status_ai,
        "Skor Flatness": f"{skor:.4f}",
        "Rekomendasi AI": keputusan_sistem,
        "Keputusan Manusia": keputusan_admin,
        "Catatan Admin": catatan
    }
    riwayat_review.append(hasil_log)

    # Format teks keluaran untuk dasbor
    output_teks = f"""
    📊 STATISTIK AI LAYER:
    -------------------------------------------
    • Status Audio: {status_ai}
    • Skor Karakteristik: {skor:.4f} (Nilai < 0.10 mengindikasikan Musik/Vektor Suara Padat)
    • Rekomendasi Awal AI: {keputusan_sistem}

    🛠️ AKSI MANUAL REVIEW LAYER (ANDA):
    -------------------------------------------
    • Status Akhir Konten: {keputusan_admin} 🔒
    • Catatan Peninjau: {catatan}
    """

    # Format tampilan log riwayat seluruh video
    log_tabel = ""
    for item in riwayat_review:
        log_tabel += f"📄 {item['Nama File']} -> AI: {item['Analisis AI']} | MANUSIA: {item['Keputusan Manusia']} ({item['Catatan Admin']})\n"

    return output_teks, log_tabel

# 4. MEMBANGUN DASBOR GRAPHICAL USER INTERFACE (GUI) DENGAN GRADIO
with gr.Blocks(title="AI + Manual Review Content Moderation") as dasbor_sistem:
    gr.Markdown("# 🎬 Dasbor Moderasi Konten: AI + Manual Review Layer")
    gr.Markdown("Sistem ini mengekstrak audio via **FFmpeg**, menganalisis karakteristik gelombang via **Librosa**, dan menyediakan jalur intervensi manual bagi Admin.")

    with gr.Row():
        with gr.Column():
            input_video = gr.Video(label="Unggah Video Konten")
            pilihan_manusia = gr.Radio(["LOLOSKAN KONTEN (APPROVE)", "BLOKIR KONTEN (REJECT)"], label="Tindakan Review Manual Anda", value="LOLOSKAN KONTEN (APPROVE)")
            catatan_admin = gr.Textbox(label="Catatan Alasan Peninjauan", placeholder="Contoh: Musik aman bebas hak cipta / Mengandung lagu berhak cipta")
            tombol_proses = gr.Button("🔥 Jalankan Pipeline AI & Simpan Keputusan", variant="primary")

        with gr.Column():
            Hasil_AI_Manusia = gr.Textbox(label="Status Hasil Sinkronisasi AI & Manusia", lines=10)
            Log_Riwayat = gr.Textbox(label="📜 Riwayat Log Moderasi Global (Database)", lines=6)

    tombol_proses.click(
        fn=proses_review_manusia,
        inputs=[input_video, pilihan_manusia, catatan_admin],
        outputs=[Hasil_AI_Manusia, Log_Riwayat]
    )

# Jalankan aplikasi sistem dengan link terowongan publik gratis
dasbor_sistem.launch(share=True, debug=True)


In [ ]:
%%writefile moderasi_otomatis.py
import os
import subprocess
import numpy as np
import librosa
import sys

def scan_audio_otomatis(video_path):
    if not os.path.exists(video_path):
        print(f"❌ Kesalahan: File video '{video_path}' tidak ditemukan.")
        return

    audio_temp = "temp_auto_extract.wav"
    if os.path.exists(audio_temp):
        os.remove(audio_temp)

    try:
        # Ekstraksi menggunakan FFmpeg secara silent background
        perintah_ffmpeg = [
            'ffmpeg', '-i', video_path, '-vn', '-acodec', 'pcm_s16le',
            '-ar', '16000', '-ac', '1', audio_temp, '-y', '-loglevel', 'quiet'
        ]
        subprocess.run(perintah_ffmpeg, check=True)

        if not os.path.exists(audio_temp) or os.path.getsize(audio_temp) == 0:
            print(f"🎬 [MUTE] {os.path.basename(video_path)} -> STATUS: OTOMATIS LOLOS (Tidak ada audio track) ✅")
            return

        # Analisis matematika sinyal dengan Librosa
        sinyal, sr = librosa.load(audio_temp, sr=16000)
        if len(sinyal) == 0:
            print(f"🎬 [MUTE] {os.path.basename(video_path)} -> STATUS: OTOMATIS LOLOS (Sinyal kosong) ✅")
            return

        # Hitung Spectral Flatness
        flatness = librosa.feature.spectral_flatness(y=sinyal)
        rata_flatness = np.mean(flatness)

        # Penentuan keputusan otomatis oleh skrip
        if rata_flatness < 0.1:
            print(f"🎬 [AUDIO/MUSIK DETECTED] {os.path.basename(video_path)}")
            print(f"   • Skor Flatness: {rata_flatness:.4f}")
            print(f"   • STATUS: MASUKKAN KE LAYER MANUAL REVIEW ⚠️")
        else:
            print(f"🎬 [BACKGROUND NOISE] {os.path.basename(video_path)} -> STATUS: OTOMATIS LOLOS ✅")

    except Exception as e:
        print(f"❌ Gagal memproses video {video_path}: {e}")
    finally:
        if os.path.exists(audio_temp):
            os.remove(audio_temp)

if __name__ == "__main__":
    # Mengambil argumen nama file video langsung dari perintah terminal
    if len(sys.argv) < 2:
        print("💡 Cara penggunaan skrip otomatis di terminal:")
        print("   python moderasi_otomatis.py nama_video.mp4")
    else:
        video_target = sys.argv[1]
        scan_audio_otomatis(video_target)


In [ ]:
import os
import glob
import shutil

# 1. Cari file video .mp4 di dalam folder penyimpanan sementara Gradio
video_temporary = glob.glob('/tmp/gradio/**/*.mp4', recursive=True)

if video_temporary:
    # Ambil video pertama yang ditemukan
    path_video_lama = video_temporary[0]
    nama_video_asli = os.path.basename(path_video_lama)

    # 2. Pindahkan video tersebut ke folder utama Colab agar terlihat di panel kiri
    path_video_baru = os.path.join('/content', nama_video_asli)
    shutil.copy(path_video_lama, path_video_baru)

    print(f"🎉 BERHASIL MENEMUKAN VIDEO!")
    print(f"Nama video Anda adalah: {nama_video_asli}")
    print(f"Sekarang file sudah dipindahkan ke folder utama panel kiri Anda.")
else:
    print("❌ Video tidak ditemukan di folder sementara.")
    print("Silakan klik ikon 'Refresh' (lingkaran panah) di panel folder kiri Colab Anda,")
    print("atau unggah ulang file video secara manual ke panel kiri tersebut.")


In [ ]:
!python moderasi_otomatis.py 20260605-214913-531.mp4


In [ ]:
# 1. Install library MediaPipe buatan Google untuk AI Vision
!pip install mediapipe opencv-python-headless -q

import cv2
import mediapipe as mp
import os
import sys

print("✅ Infrastruktur AI Computer Vision & MediaPipe siap digunakan!")

# 2. SEBUAH SKRIP MANDIRI: Filter Faceless Video Otomatis
%%writefile filter_faceless.py
import cv2
import mediapipe as mp
import os
import sys

def deteksi_faceless_video(video_path):
    if not os.path.exists(video_path):
        print(f"❌ Kesalahan: File video '{video_path}' tidak ditemukan.")
        return

    # Inisialisasi AI Detektor Wajah dari MediaPipe
    mp_face_detection = mp.solutions.face_detection

    # Membuka file video menggunakan OpenCV
    kap_video = cv2.VideoCapture(video_path)

    # Mengambil informasi total frame dan FPS video untuk laporan
    total_frames = int(kap_video.get(cv2.CAP_PROP_FRAME_COUNT))
    fps = kap_video.get(cv2.CAP_PROP_FPS)

    wajah_terdeteksi = False
    frame_terpindai = 0
    lompat_frame = 5 # Trik optimasi: scan setiap 5 frame sekali agar proses super cepat

    # Jalankan model AI dengan tingkat kepercayaan minimum 50% (0.5)
    with mp_face_detection.FaceDetection(model_selection=0, min_detection_confidence=0.5) as face_detection:
        while kap_video.isOpened():
            sukses, frame = kap_video.read()
            if not sukses:
                break

            # Jalankan deteksi hanya pada interval frame yang ditentukan (efisiensi memori)
            if frame_terpindai % lompat_frame == 0:
                # MediaPipe membutuhkan format warna RGB, sedangkan OpenCV menggunakan BGR
                frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

                # Proses frame menggunakan AI
                hasil_ai = face_detection.process(frame_rgb)

                # Jika hasil deteksi wajah ditemukan, kunci status dan hentikan pencarian
                if hasil_ai.detections:
                    wajah_terdeteksi = True
                    break

            frame_terpindai += 1

    kap_video.release()

    # 3. KEPUTUSAN LOGIKA AKHIR FILTER
    nama_file = os.path.basename(video_path)
    print("\n📊 HASIL FILTER AI COMPUTER VISION:")
    print("--------------------------------------------------")
    print(f"• Nama File Video : {nama_file}")
    print(f"• Total Frame     : {total_frames} frame (~{total_frames/fps:.2f} detik)")

    if wajah_terdeteksi:
        print("• Kategori Video  : 👤 HUMAN FACE VIDEO (Mengandung Wajah Manusia)")
        print("• Status Filter   : KONTEN DITOLAK / PERLU REDIREKSI ❌")
    else:
        print("• Kategori Video  : ✅ FACELESS VIDEO (Murni Tanpa Wajah Manusia)")
        print("• Status Filter   : OTOMATIS LOLOS FILTER KONTEN NYAMAN 🚀")
    print("--------------------------------------------------")

if __name__ == "__main__":
    if len(sys.argv) < 2:
        print("💡 Cara penggunaan skrip otomatis di terminal:")
        print("   python filter_faceless.py nama_video.mp4")
    else:
        video_target = sys.argv[1]
        deteksi_faceless_video(video_target)


In [ ]:
!pip install mediapipe opencv-python-headless -q
print("✅ Infrastruktur AI Computer Vision & MediaPipe siap digunakan!")


In [ ]:
%%writefile filter_faceless.py
import cv2
import mediapipe as mp
import os
import sys

def deteksi_faceless_video(video_path):
    if not os.path.exists(video_path):
        print(f"❌ Kesalahan: File video '{video_path}' tidak ditemukan.")
        return

    # Inisialisasi AI Detektor Wajah dari MediaPipe
    mp_face_detection = mp.solutions.face_detection

    # Membuka file video menggunakan OpenCV
    kap_video = cv2.VideoCapture(video_path)

    # Mengambil informasi total frame dan FPS video
    total_frames = int(kap_video.get(cv2.CAP_PROP_FRAME_COUNT))
    fps = kap_video.get(cv2.CAP_PROP_FPS)

    wajah_terdeteksi = False
    frame_terpindai = 0
    lompat_frame = 5 # Scan setiap 5 frame sekali agar proses super cepat

    # Jalankan model AI dengan tingkat kepercayaan minimum 50%
    with mp_face_detection.FaceDetection(model_selection=0, min_detection_confidence=0.5) as face_detection:
        while kap_video.isOpened():
            sukses, frame = kap_video.read()
            if not sukses:
                break

            if frame_terpindai % lompat_frame == 0:
                # MediaPipe membutuhkan format warna RGB
                frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

                # Proses frame menggunakan AI
                hasil_ai = face_detection.process(frame_rgb)

                # Jika wajah ditemukan, kunci status dan hentikan pencarian
                if hasil_ai.detections:
                    wajah_terdeteksi = True
                    break

            frame_terpindai += 1

    kap_video.release()

    # KEPUTUSAN LOGIKA AKHIR FILTER
    nama_file = os.path.basename(video_path)
    print("\n📊 HASIL FILTER AI COMPUTER VISION:")
    print("--------------------------------------------------")
    print(f"• Nama File Video : {nama_file}")
    if fps > 0:
        print(f"• Total Frame     : {total_frames} frame (~{total_frames/fps:.2f} detik)")
    else:
        print(f"• Total Frame     : {total_frames} frame")

    if wajah_terdeteksi:
        print("• Kategori Video  : 👤 HUMAN FACE VIDEO (Mengandung Wajah Manusia)")
        print("• Status Filter   : KONTEN DITOLAK / PERLU REDIREKSI ❌")
    else:
        print("• Kategori Video  : ✅ FACELESS VIDEO (Murni Tanpa Wajah Manusia)")
        print("• Status Filter   : OTOMATIS LOLOS FILTER KONTEN NYAMAN 🚀")
    print("--------------------------------------------------")

if __name__ == "__main__":
    if len(sys.argv) < 2:
        print("💡 Cara penggunaan skrip otomatis di terminal:")
        print("   python filter_faceless.py nama_video.mp4")
    else:
        video_target = sys.argv[1]
        deteksi_faceless_video(video_target)


In [ ]:
!python filter_faceless.py 20260605-214913-531.mp4


In [ ]:
%%writefile filter_faceless.py
import cv2
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks import vision
import os
import sys
import urllib.request

def deteksi_faceless_video(video_path):
    if not os.path.exists(video_path):
        print(f"❌ Kesalahan: File video '{video_path}' tidak ditemukan.")
        return

    # 1. Unduh file model BlazeFace resmi jika belum ada di server Colab
    model_path = "blaze_face_short_range.tflite"
    if not os.path.exists(model_path):
        print("⏳ Sedang mengunduh model AI Face Detection...")
        url_model = "https://googleapis.com"
        urllib.request.urlretrieve(url_model, model_path)

    # 2. Konfigurasi MediaPipe Tasks API standar terbaru
    base_options = python.BaseOptions(model_asset_path=model_path)
    options = vision.FaceDetectorOptions(base_options=base_options, min_detection_confidence=0.5)

    # Membuka file video menggunakan OpenCV
    kap_video = cv2.VideoCapture(video_path)
    total_frames = int(kap_video.get(cv2.CAP_PROP_FRAME_COUNT))
    fps = kap_video.get(cv2.CAP_PROP_FPS)

    wajah_terdeteksi = False
    frame_terpindai = 0
    lompat_frame = 5 # Trik optimasi: scan setiap 5 frame sekali

    # 3. Jalankan detektor wajah dari modul vision
    with vision.FaceDetector.create_from_options(options) as detector:
        while kap_video.isOpened():
            sukses, frame = kap_video.read()
            if not sukses:
                break

            if frame_terpindai % lompat_frame == 0:
                # Ubah BGR bawaan OpenCV menjadi RGB untuk MediaPipe
                frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

                # Ubah matriks gambar menjadi objek mp.Image resmi
                mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=frame_rgb)

                # Proses deteksi visual
                hasil_deteksi = detector.detect(mp_image)

                # Jika ada wajah terdeteksi, kunci status dan hentikan pemindaian
                if hasil_deteksi.detections:
                    wajah_terdeteksi = True
                    break

            frame_terpindai += 1

    kap_video.release()

    # 4. LAPORAN LOGIKA FILTER AKHIR
    nama_file = os.path.basename(video_path)
    print("\n📊 HASIL FILTER AI COMPUTER VISION (MIGRATED TO TASKS API):")
    print("--------------------------------------------------")
    print(f"• Nama File Video : {nama_file}")
    if fps > 0:
        print(f"• Total Frame     : {total_frames} frame (~{total_frames/fps:.2f} detik)")
    else:
        print(f"• Total Frame     : {total_frames} frame")

    if wajah_terdeteksi:
        print("• Kategori Video  : 👤 HUMAN FACE VIDEO (Mengandung Wajah Manusia)")
        print("• Status Filter   : KONTEN DITOLAK / PERLU REDIREKSI ❌")
    else:
        print("• Kategori Video  : ✅ FACELESS VIDEO (Murni Tanpa Wajah Manusia)")
        print("• Status Filter   : OTOMATIS LOLOS FILTER KONTEN NYAMAN 🚀")
    print("--------------------------------------------------")

if __name__ == "__main__":
    if len(sys.argv) < 2:
        print("💡 Cara penggunaan skrip otomatis di terminal:")
        print("   python filter_faceless.py nama_video.mp4")
    else:
        video_target = sys.argv[1]
        deteksi_faceless_video(video_target)


In [ ]:
!python filter_faceless.py 20260605-214913-531.mp4


In [ ]:
%%writefile filter_faceless.py
import cv2
import mediapipe as mp
# PERBAIKAN JALUR IMPORT: Memanggil modul vision melalui sub-modul python resmi
from mediapipe.tasks import python
from mediapipe.tasks.python import vision
import os
import sys
import urllib.request

def deteksi_faceless_video(video_path):
    if not os.path.exists(video_path):
        print(f"❌ Kesalahan: File video '{video_path}' tidak ditemukan.")
        return

    # 1. Unduh file model BlazeFace jika belum ada di server Colab
    model_path = "blaze_face_short_range.tflite"
    if not os.path.exists(model_path):
        print("⏳ Sedang mengunduh model AI Face Detection...")
        url_model = "https://storage.googleapis.com/mediapipe-models/face_detector/blaze_face_short_range/float16/1/blaze_face_short_range.tflite"
        urllib.request.urlretrieve(url_model, model_path)

    # 2. Konfigurasi MediaPipe Tasks API standar terbaru
    base_options = python.BaseOptions(model_asset_path=model_path)
    options = vision.FaceDetectorOptions(base_options=base_options, min_detection_confidence=0.5)

    # Membuka file video menggunakan OpenCV
    kap_video = cv2.VideoCapture(video_path)
    total_frames = int(kap_video.get(cv2.CAP_PROP_FRAME_COUNT))
    fps = kap_video.get(cv2.CAP_PROP_FPS)

    wajah_terdeteksi = False
    frame_terpindai = 0
    lompat_frame = 5 # Scan setiap 5 frame sekali agar proses kilat

    # 3. Jalankan detektor wajah dari modul vision terbaru
    with vision.FaceDetector.create_from_options(options) as detector:
        while kap_video.isOpened():
            sukses, frame = kap_video.read()
            if not sukses:
                break

            if frame_terpindai % lompat_frame == 0:
                # Ubah format warna BGR OpenCV menjadi RGB untuk MediaPipe
                frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

                # Ubah matriks gambar menjadi objek mp.Image
                mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=frame_rgb)

                # Proses deteksi wajah
                hasil_deteksi = detector.detect(mp_image)

                # Jika komponen wajah ditemukan, kunci status dan hentikan pencarian
                if hasil_deteksi.detections:
                    wajah_terdeteksi = True
                    break

            frame_terpindai += 1

    kap_video.release()

    # 4. LAPORAN LOGIKA FILTER AKHIR
    nama_file = os.path.basename(video_path)
    print("\n📊 HASIL FILTER AI COMPUTER VISION (TASKS API FIXED):")
    print("--------------------------------------------------")
    print(f"• Nama File Video : {nama_file}")
    if fps > 0:
        print(f"• Total Frame     : {total_frames} frame (~{total_frames/fps:.2f} detik)")
    else:
        print(f"• Total Frame     : {total_frames} frame")

    if wajah_terdeteksi:
        print("• Kategori Video  : 👤 HUMAN FACE VIDEO (Mengandung Wajah Manusia)")
        print("• Status Filter   : KONTEN DITOLAK / PERLU REDIREKSI ❌")
    else:
        print("• Kategori Video  : ✅ FACELESS VIDEO (Murni Tanpa Wajah Manusia)")
        print("• Status Filter   : OTOMATIS LOLOS FILTER KONTEN NYAMAN 🚀")
    print("--------------------------------------------------")

if __name__ == "__main__":
    if len(sys.argv) < 2:
        print("💡 Cara penggunaan skrip otomatis di terminal:")
        print("   python filter_faceless.py nama_video.mp4")
    else:
        video_target = sys.argv[1]
        deteksi_faceless_video(video_target)


In [ ]:
!python filter_faceless.py 20260605-214913-531.mp4


In [ ]:
import cv2
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision
import os
import gradio as gr
import urllib.request

# 1. Pastikan model BlazeFace sudah terunduh
model_path = "blaze_face_short_range.tflite"
if not os.path.exists(model_path):
    print("⏳ Sedang mengunduh model AI Face Detection...")
    url_model = "https://googleapis.com"
    urllib.request.urlretrieve(url_model, model_path)

# 2. Fungsi Utama untuk Backend Gradio
def filter_video_gradio(video_input):
    if video_input is None:
        return "Silakan unggah file video terlebih dahulu.", ""

    # Konfigurasi MediaPipe
    base_options = python.BaseOptions(model_asset_path=model_path)
    options = vision.FaceDetectorOptions(base_options=base_options, min_detection_confidence=0.5)

    kap_video = cv2.VideoCapture(video_input)
    total_frames = int(kap_video.get(cv2.CAP_PROP_FRAME_COUNT))
    fps = kap_video.get(cv2.CAP_PROP_FPS)

    wajah_terdeteksi = False
    frame_terpindai = 0
    lompat_frame = 5 # Melompati frame agar proses web cepat dan responsif

    with vision.FaceDetector.create_from_options(options) as detector:
        while kap_video.isOpened():
            sukses, frame = kap_video.read()
            if not sukses:
                break

            if frame_terpindai % lompat_frame == 0:
                frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=frame_rgb)
                hasil_deteksi = detector.detect(mp_image)

                if hasil_deteksi.detections:
                    wajah_terdeteksi = True
                    break

            frame_terpindai += 1

    kap_video.release()

    # 3. Penyusunan Output untuk Ditampilkan ke Web UI
    durasi = total_frames / fps if fps > 0 else 0
    info_video = f"• Total Frame: {total_frames} frame\n• Estimasi Durasi: {durasi:.2f} detik"

    if wajah_terdeteksi:
        hasil_kategori = "👤 HUMAN FACE VIDEO\n(Mengandung Wajah Manusia - KONTEN DITOLAK ❌)"
    else:
        hasil_kategori = "✅ FACELESS VIDEO\n(Murni Tanpa Wajah - OTOMATIS LOLOS 🚀)"

    return hasil_kategori, info_video

# 4. Mendesain Tampilan Antarmuka Grafis (Gradio UI)
with gr.Blocks(title="AI Faceless Video Filter") as web_filter:
    gr.Markdown("# 🎬 AI Faceless Video Filter & Moderator")
    gr.Markdown("Unggah video Anda untuk memfilter secara otomatis apakah konten tersebut bersih dari wajah manusia menggunakan **Google MediaPipe Tasks API**.")

    with gr.Row():
        with gr.Column():
            input_video = gr.Video(label="Unggah Video Konten (.mp4)")
            tombol_filter = gr.Button("🔍 Scan Konten dengan AI", variant="primary")

        with gr.Column():
            output_kategori = gr.Textbox(label="Hasil Kategori & Status Filter AI", lines=3)
            output_info = gr.Textbox(label="Statistik Metadata Video", lines=3)

    tombol_filter.click(
        fn=filter_video_gradio,
        inputs=input_video,
        outputs=[output_kategori, output_info]
    )

# 5. Luncurkan Server dan Terbitkan Tautan Publik
web_filter.launch(share=True, debug=True)
